# TakeOff.ai — Train the spaces (room) model

Run this on a **GPU** runtime (Colab: *Runtime → Change runtime type → GPU*; or RunPod/Jupyter).
It downloads CubiCasa5K, trains a YOLOv8-seg model, checks it against the accuracy gate,
and gives you `best.pt` to drop into the backend at `models/best.pt`.

The app already knows how to use it — once `best.pt` exists, raster AI stops raising
`ModelUnavailableError` and returns real detections, with zero code change.


## 1. Confirm you have a GPU


In [ ]:
!nvidia-smi -L || echo 'NO GPU — switch the runtime to GPU before continuing'


## 2. Get the code
If the repo is **private**, set a GitHub token first (a fine-grained PAT with read access):


In [ ]:
import os
# os.environ['GITHUB_TOKEN'] = 'ghp_xxx'   # uncomment + set for a private repo
REPO = 'github.com/Siddartha-DevOps/TakeOff.git'
url = f"https://{os.environ['GITHUB_TOKEN']}@{REPO}" if os.environ.get('GITHUB_TOKEN') else f'https://{REPO}'
!git clone $url
%cd TakeOff/app/backend


## 3. Install the ML stack
torch with the CUDA wheel, then the pinned training deps.


In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements-ml.txt
!python -m ml.preflight   # shows what's present/missing


## 4. Dataset — CubiCasa5K → versioned YOLO-seg
~5 GB download; converts to the trainer's format automatically.


In [ ]:
import datetime
TS = datetime.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
!python -c "from ml.datasets.acquire_cubicasa import download_cubicasa; download_cubicasa('cubicasa5k.zip')"
!mkdir -p data/cubicasa5k && unzip -q -o cubicasa5k.zip -d data/cubicasa5k
!python -m ml.datasets.acquire_cubicasa --root data/cubicasa5k --out data/spaces_v1 --created-at $TS
!python -m ml.preflight --data data/spaces_v1/data.yaml --require train


## 5. Smoke run (1 epoch) — verifies the whole pipeline before the long train


In [ ]:
!python -m ml.training.run_training --data data/spaces_v1/data.yaml --task spaces --smoke --no-promote


## 6. Full training
Defaults: yolov8m-seg, 100 epochs, imgsz 1280 (large sheets). Adjust EPOCHS if you're impatient.


In [ ]:
!python -m ml.training.run_training --data data/spaces_v1/data.yaml --task spaces --epochs 100 --imgsz 1280
# writes weights to models/best.pt


## 7. Prove accuracy (the gate)
Runs the model over the val split and scores mIoU / mAP / measurement-error vs the promotion gate.


In [ ]:
!python -m ml.eval.predict_golden --dataset data/spaces_v1 --weights models/best.pt --evaluate --out preds.json || true
!python -m ml.eval.build_golden --dataset data/spaces_v1 --out golden.json --predictions preds.json
!python -m ml.eval.report --golden golden.json --out accuracy_report.md || true
print(open('accuracy_report.md').read())


## 8. Download `best.pt`
Put this file at `app/backend/models/best.pt` on your backend (Render: mount a volume or bake it in),
restart the API, and raster AI is live. To label your own plans for higher accuracy, see `ml/TRAINING.md`.


In [ ]:
try:
    from google.colab import files
    files.download('models/best.pt')
except Exception:
    print('Not on Colab — copy models/best.pt off this box manually.')
